In [ ]:
import pandas as pd
import psycopg2
import pandas as pd
import geopandas as gpd
import requests
from geopandas.tools import sjoin
import time
import datetime
from collections import Counter 
import re
import random
import numpy as np
import os



In [ ]:
liste_biais = [["RESIDENCE",3],["CHEZ",2],["BAT",5],["APPT",5],["MME",3],["MR",4],["RES",3],["HOPITAL",3],["MAISON",4],["RETRAITE",6],["CENTRE",4],["HOTEL",5],["QUARTIER",2]]
liste_nom_propre = ["CHARLES DE GAULLES", "JEAN MOULIN", "MARIE CURIE", "Foch"]

def add_bruit_unique(df, biais, pos_bruit, taux_bruit):

    df_bruite = df.copy() 

    #df_bruite = df_bruite.reset_index(drop=True)

    for i in df_bruite.index:

        adresse_part = str(df_bruite.at[i,"adresse"]).split(" ")
        adresse_part = ' '.join(adresse_part).split() ## suppression str vides 

        if pos_bruit > len(adresse_part):
            adresse_part.insert(len(adresse_part),biais)
        else :
            adresse_part.insert(pos_bruit,biais)
        df_bruite.at[i, "adresse"] = " ".join(adresse_part)

    return df_bruite


def add_bruit_unique_joint(df, biais, pos_bruit, taux_bruit):

    df_bruite = df.copy() 

    #df_bruite = df_bruite.reset_index(drop=True)

    for i in df_bruite.index:

        adresse_part = str(df_bruite.at[i,"adresse"]).split(" ")
        adresse_part = ' '.join(adresse_part).split() ## suppression str vides 

        ## Génération aléatoire du nom propre : 
        j = random.randint(0,3)
        biais_np = biais + liste_nom_propre[j]

        if pos_bruit > len(adresse_part):
            adresse_part.insert(len(adresse_part),biais_np)
        else :
            adresse_part.insert(pos_bruit,biais_np)
        df_bruite.at[i, "adresse"] = " ".join(adresse_part)

    return df_bruite

In [ ]:
df_ref_ini = pd.read_csv("./ecoles/fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv", sep=";")

col_to_keep=['numero_uai','adresse_uai','code_postal_uai','libelle_commune','coordonnee_x', 'coordonnee_y', 'epsg', 'latitude',
       'longitude', 'appariement', 'localisation','code_departement', 'code_region']
df_ref_ini = df_ref_ini[col_to_keep]

reg_metrop = [11,24,27,28,32,44,54,53,75,76,84,93,94]
df_ref_ini = df_ref_ini[df_ref_ini["code_region"].isin(reg_metrop)]
df_ref_ini.rename(columns={'adresse_uai': 'adresse'}, inplace=True)

df_ref_ini = df_ref_ini.sample(n=1000).reset_index()


# Dictionnaire pour stocker les DataFrames
df_ref_biais = {}

# Boucle pour appliquer le bruit à chaque biais
for biais, pos_bruit in liste_biais:
    df_ref_biais[f"df_ref_{biais}"] = add_bruit_unique(df_ref_ini,biais,pos_bruit, 1)
    df_ref_biais[f"df_ref_{biais}_np"] = add_bruit_unique_joint(df_ref_ini,biais,pos_bruit, 1)

In [ ]:
def ref_to_geocoding(df):
    df["requete"] =df['adresse'] + " " + str(df["code_postal_uai"])+ " " + df["libelle_commune"]
    return df 


for nom_df, df in df_ref_biais.items():
    df_modifie = ref_to_geocoding(df)
    df_ref_biais[nom_df] = df_modifie
    print(f"DataFrame: {nom_df} prêt à être géocodé")
    print("---")

In [ ]:
def geocode(df,liste_biais, join = False):
    """Geocode dataframe with Etalab addok"""
    name =[x for x in globals() if globals()[x] is df][0]
    print(f"Proceed to geocode on address for {name}...")
    
    for i in df.index:
        try:
            # get json response
            r = requests.get('https://addok-data.curie.net/search?q='+df["requete"][i])
            response = r.json()

            if i%10000==0 : 
                print(f"Proceed geocoding at the {i}th row")
            if response["features"]!=[]:
                # parse json to insert value in dataframe
                df.at[i, 'x'] = str(response["features"][0]["geometry"]["coordinates"][0])
                df.at[i, 'y'] = str(response["features"][0]["geometry"]["coordinates"][1])
                df.at[i, 'score'] = str(response["features"][0]["properties"]["score"])

                if float(df["score"][i])<0.4:
                    df.at[i, 'trust_score'] = 'low'
                elif float(df["score"][i])>0.4 and float(df["score"][i])<0.65:
                    df.at[i, 'trust_score'] = 'middle'
                elif float(df["score"][i])>0.65 and float(df["score"][i])<0.9:
                    df.at[i, 'trust_score'] = 'middle'
                else:
                    df.at[i, 'trust_score'] = 'high'

                df.at[i, 'street'] = str(response["features"][0]["properties"]["name"]).replace("'", " ")
                df.at[i, 'city'] = str(response["features"][0]["properties"]["city"]).replace("'", " ")
                df.at[i, 'pc_city'] = str(response["features"][0]["properties"]["postcode"])
                df.at[i, 'ic_city'] = str(response["features"][0]["properties"]["citycode"])

                context = (str(response["features"][0]["properties"]["context"]).replace("'", " ")).split(",")
                df.at[i, 'code_dept'] = context[0]
                df.at[i, 'dept'] = context[1]

                if len(df["code_dept"][i])==2:
                    df.at[i, 'reg'] = context[2]
                else:
                    df.at[i, 'reg'] = "other"

                df.at[i, "address"] = str(response["features"][0]["properties"]["label"]).replace("'", " ")
                
                if (df["adresse"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_init'] = "true"
                else:
                    df.at[i, 'address_has_num_init'] = "false"
                    
                if (df["street"][i].lstrip())[0].isdigit():
                    df.at[i, 'address_has_num_geoloc'] = "true"
                else:
                    df.at[i, 'address_has_num_geoloc'] = "false"
                    
                if df["codepost"][i] == df["pc_city"][i]:
                    df.at[i, 'same_city'] = "true"
                else:
                    df.at[i, 'same_city'] = "false" 

                ##Analyse biais : 
                for mot in liste_biais: 
                    if join : 
                        if mot[0] and mot[1] in df["adresse"].str.split(" ") : 
                            df.at[i,f"adresse_has_{mot[0]}_{mot[1]}_init"] = True 
                    else : 
                        if mot in df["adresse"].str.split(' '):
                            df.at[i,f"adresse_has_{mot}_init"] = True 
                        else : 
                            df.at[i,f"adresse_has_{mot}_init"] = False 

                    
                ## Check up si mot "hotel" et "chez" sont présent dans la réponse du géocodage?     
                # if "hotel" in df["street"][i] or "hôtel" in df["street"][i]: 
                #     df.at[i, 'hostel'] = "true"
                # else:
                #     df.at[i, 'hostel'] = "false"

                # if "chez" in df["street"][i]: 
                #     df.at[i, 'hosted'] = "true"
                # else:
                #     df.at[i, 'hosted'] = "false"
                    
                df.at[i, 'date_geoloc'] = str(datetime.date.today())
                #df.at[i, 'etalab_version'] = str(response["licence"])
                #df.at[i, 'ban_version'] = "2021-04-27"
            else:
                pass
        except:
            pass
    return df

def spatialjoin(df, df_iris, df_epci, df_dept):
    """Spatial join of the database and the iris/epci layers"""
    name =[x for x in globals() if globals()[x] is df][0]
    print(f"Perfom to spatial join on IRIS and EPCI layers for {name}...")
    gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=2154)

    df_join_iris = gpd.sjoin(gdf, df_iris[['CODE_IRIS','geometry']], how="left", op='within')
    df_join_iris.drop('index_right', axis=1, inplace=True)

    df_epci =df_epci.to_crs(epsg=2154)
    df_join_iris_epci = gpd.sjoin(df_join_iris, df_epci[['CODE_EPCI','geometry']], how="left", op='within')
    df_join_iris_epci.drop('index_right', axis=1, inplace=True)

    df_join_iris_epci_dept = gpd.sjoin(df_join_iris_epci, df_dept[['CODE_DEPT','geometry']], how="left", op='within')
    df_join_iris_epci_dept.drop('index_right', axis=1, inplace=True)
    return df_join_iris_epci_dept

df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')
df_epci = gpd.read_file('H:/canc_air/data/zones_geographiques/epci/EPCI_SHAPEFILE.shp')
df_dept = gpd.read_file('H:/canc_air/data/zones_geographiques/departements/DEPARTEMENT.shp')

In [ ]:
df_ref_biais_geocoded = {}

for nom_df, df in df_ref_biais.items():
    df_geocoded = geocode(df,liste_biais)
    print(f"{nom_df} géocodé")
    df_geo_spatial = spatialjoin(df_geocoded, df_iris, df_epci, df_dept)
    df_ref_biais_geocoded[nom_df] = df_geo_spatial
    df_geo_spatial.to_csv(f"../geocodeur/data_with_biais/ref_biaised/{nom_df}.csv",sep=";")
    print(f"{nom_df} jointure spatiale faite")
    print("---")